In [12]:
import ee
import geemap.core as geemap
import pandas as pd
ee.Authenticate()
ee.Initialize(project='firedatatest')

In [13]:
#narrow our data to only look at colorado
colorado = ee.FeatureCollection("TIGER/2018/States").filter("NAME == 'Colorado'").first().geometry() #selects colorado geometry from all the states
gridmet = ee.ImageCollection("IDAHO_EPSCOR/GRIDMET").filterBounds(colorado) #only collect images from colorado
firms = ee.ImageCollection("FIRMS").filterBounds(colorado) #only collect images from colorado
srtm = ee.Image("USGS/SRTMGL1_003").clip(colorado) #clip elevation data to only be colorado (clip for image is like using filter for imageCollection)

In [14]:
#Dynamix data - changes every day
#helpful enviornmental conditions for fire prediction
precipitation = gridmet.select('pr') #precipitation amount: 0-690.44 mm
max_temperature = gridmet.select('tmmx') #maximum temperature: 233.08-327.14 K
min_humidity = gridmet.select('rmin') #minimum relative humidity: 0-100%
wind_speed = gridmet.select('vs') #wind velocity at 10m: 0.14-29.13 m/s
potential_fuel = gridmet.select('erc') #energy release component: 0-131.85 (fire danger index)

#fire data
fire_data = firms.select('T21') #brightness temperature: 300-509.29 K

#static data - does not change
elevation = srtm.select('elevation') #elevation: -10-6500 m
terrain = ee.Terrain.products(elevation).select(['elevation', 'slope', 'aspect']) #calculates slope, and aspect, adding it to the elevation data